In [ ]:
import sys
from pathlib import Path
from typing import Any, Sequence

import json
import csv
import numpy as np
import pandas as pd
import polars as pl
from IPython.display import display
from sklearn.metrics import roc_auc_score

notebook_dir = Path.cwd().resolve()
example_root = notebook_dir.parent if notebook_dir.name == "notebooks" else notebook_dir
project_root = example_root.parents[1]

if str(example_root) not in sys.path:
    sys.path.insert(0, str(example_root))
if str(project_root / "src") not in sys.path:
    sys.path.insert(0, str(project_root / "src"))

import mars
from mars.analysis import profile_risk, profile_stats
from mars.feature import MarsLiteOptBinner, MarsLinearSelector, MarsStatsSelector
from mars.analysis.missing_shift import (
    MarsMissingShiftConfig,
    MarsMissingShiftResult,
    MarsMissingShiftScanner,
)


from config import CONFIG

pl.Config.set_tbl_cols(200)
mars.set_log_level("info")
print(mars.__version__)

# 交付产物独立存放，顺序执行不读取历史报告。
OUTPUT_DIR = project_root / ".artifacts" / "notebook-repair" / "home-credit"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# **1. 数据预处理**

In [ ]:
df_train = pl.read_csv(CONFIG.RAW_DATA / "application_train.csv")
df_test = pl.read_csv(CONFIG.RAW_DATA / "application_test.csv") 

In [ ]:
df_train.shape, df_test.shape

In [ ]:
48744 / 307511

In [ ]:
# 本地逐行调试可自行调用 head；公开示例仅输出规模。
df_train.shape

In [ ]:
df_test.shape

In [ ]:
df_test = df_test.with_columns(pl.lit(None).alias("TARGET"))
df_test.shape

In [ ]:
df = pl.concat([df_train, df_test.select(df_train.columns)], how="vertical_relaxed")

In [ ]:
df.shape

In [ ]:
df = df.with_columns(
    pl.when(pl.col("TARGET") >= 0)
    .then(
        pl.date(2026, 1, 1)
        + pl.duration(
            days=(pl.col("SK_ID_CURR").hash(seed=42) % 181).cast(pl.Int64)
        )
    )
    .otherwise(
        pl.date(2026, 7, 1)
        + pl.duration(
            days=(pl.col("SK_ID_CURR").hash(seed=43) % 31).cast(pl.Int64)
        )
    )
    .alias("APPLY_DATE")
)

train_dates = (
    df.filter(pl.col("TARGET") >= 0)
    .get_column("APPLY_DATE")
    .sort()
)

n = len(train_dates)
assert n > 90_000

val_start = train_dates[n - 90_000]
test_start = train_dates[n - 60_000]
oot_start = train_dates[n - 30_000]

df = (
    df.with_columns(
        pl.when(pl.col("TARGET").is_null())
        .then(pl.lit("4_oot_undue"))
        .when(pl.col("APPLY_DATE") < val_start)
        .then(pl.lit("0train"))
        .when(pl.col("APPLY_DATE") < test_start)
        .then(pl.lit("1val"))
        .when(pl.col("APPLY_DATE") < oot_start)
        .then(pl.lit("2test"))
        .otherwise(pl.lit("3oot"))
        .alias("dataset_flag")
    )
    .sort(["APPLY_DATE", "SK_ID_CURR"])
)

print(
    df.group_by("dataset_flag")
    .agg(
        pl.len().alias("count"),
        pl.col("APPLY_DATE").min().alias("min_date"),
        pl.col("APPLY_DATE").max().alias("max_date"),
        pl.col("TARGET").eq(1).sum().alias("bad_count"),
    )
    .sort("dataset_flag")
)

# **2. Config**

In [ ]:
df.schema

In [ ]:
id_col = "SK_ID_CURR"
y = "TARGET"
dt = "APPLY_DATE"
amt = "AMT_CREDIT"
seg = "dataset_flag"
# 保持原候选范围，按字段角色排除 ID、标签、日期和分区。
fcols = [feature for feature in df.columns if feature not in {id_col, y, dt, seg}]
print(f"候选特征数 {len(fcols)}")

In [ ]:
# Curated labels for the non-repeating application features.
FEATURE_NAMES_CN = {
    "NAME_CONTRACT_TYPE": "贷款合同类型",
    "CODE_GENDER": "申请人性别",
    "FLAG_OWN_CAR": "是否有车",
    "FLAG_OWN_REALTY": "是否有房产",
    "CNT_CHILDREN": "子女数量",
    "AMT_INCOME_TOTAL": "申请人总收入",
    "AMT_CREDIT": "申请贷款金额",
    "AMT_ANNUITY": "贷款每期应还金额",
    "AMT_GOODS_PRICE": "贷款对应商品价格",
    "NAME_TYPE_SUITE": "申请陪同人员类型",
    "NAME_INCOME_TYPE": "收入类型",
    "NAME_EDUCATION_TYPE": "最高教育程度",
    "NAME_FAMILY_STATUS": "婚姻家庭状态",
    "NAME_HOUSING_TYPE": "居住住房类型",
    "REGION_POPULATION_RELATIVE": "居住地区相对人口规模（标准化）",
    "DAYS_BIRTH": "年龄（相对申请日天数）",
    "DAYS_EMPLOYED": "当前工作开始距申请日天数",
    "DAYS_REGISTRATION": "登记信息变更距申请日天数",
    "DAYS_ID_PUBLISH": "身份证件变更距申请日天数",
    "OWN_CAR_AGE": "车辆车龄",
    "FLAG_MOBIL": "是否提供手机号",
    "FLAG_EMP_PHONE": "是否提供工作电话",
    "FLAG_WORK_PHONE": "是否提供家庭电话（WORK_PHONE字段）",
    "FLAG_CONT_MOBILE": "手机是否可联系",
    "FLAG_PHONE": "是否提供家庭电话",
    "FLAG_EMAIL": "是否提供邮箱",
    "OCCUPATION_TYPE": "职业类型",
    "CNT_FAM_MEMBERS": "家庭成员数量",
    "REGION_RATING_CLIENT": "居住地区评级",
    "REGION_RATING_CLIENT_W_CITY": "考虑城市因素的地区评级",
    "WEEKDAY_APPR_PROCESS_START": "申请星期",
    "HOUR_APPR_PROCESS_START": "申请小时",
    "REG_REGION_NOT_LIVE_REGION": "登记地区与居住地区是否不同",
    "REG_REGION_NOT_WORK_REGION": "登记地区与工作地区是否不同",
    "LIVE_REGION_NOT_WORK_REGION": "居住地区与工作地区是否不同",
    "REG_CITY_NOT_LIVE_CITY": "登记城市与居住城市是否不同",
    "REG_CITY_NOT_WORK_CITY": "登记城市与工作城市是否不同",
    "LIVE_CITY_NOT_WORK_CITY": "居住城市与工作城市是否不同",
    "ORGANIZATION_TYPE": "工作单位组织类型",
    "EXT_SOURCE_1": "外部数据评分1",
    "EXT_SOURCE_2": "外部数据评分2",
    "EXT_SOURCE_3": "外部数据评分3",
    "FONDKAPREMONT_MODE": "住房维修基金类型（众数口径）",
    "HOUSETYPE_MODE": "住房建筑类型（众数口径）",
    "TOTALAREA_MODE": "住房总面积（众数口径）",
    "WALLSMATERIAL_MODE": "住房墙体材料（众数口径）",
    "EMERGENCYSTATE_MODE": "住房紧急状态（众数口径）",
    "OBS_30_CNT_SOCIAL_CIRCLE": "社交圈30天逾期可观察样本数",
    "DEF_30_CNT_SOCIAL_CIRCLE": "社交圈30天逾期样本数",
    "OBS_60_CNT_SOCIAL_CIRCLE": "社交圈60天逾期可观察样本数",
    "DEF_60_CNT_SOCIAL_CIRCLE": "社交圈60天逾期样本数",
    "DAYS_LAST_PHONE_CHANGE": "最近更换电话距申请日天数",
    "AMT_REQ_CREDIT_BUREAU_HOUR": "申请前1小时征信查询次数",
    "AMT_REQ_CREDIT_BUREAU_DAY": "申请前1天征信查询次数（不含近1小时）",
    "AMT_REQ_CREDIT_BUREAU_WEEK": "申请前1周征信查询次数（不含近1天）",
    "AMT_REQ_CREDIT_BUREAU_MON": "申请前1个月征信查询次数（不含近1周）",
    "AMT_REQ_CREDIT_BUREAU_QRT": "申请前3个月征信查询次数（不含近1个月）",
    "AMT_REQ_CREDIT_BUREAU_YEAR": "申请前1年征信查询次数（不含近3个月）",
}

# Original spelling is preserved: YEARS_BEGINEXPLUATATION is misspelled in Kaggle.
HOUSING_PARTS_CN = {
    "APARTMENTS": "公寓指标",
    "BASEMENTAREA": "地下室面积",
    "YEARS_BEGINEXPLUATATION": "建筑开始使用年份",
    "YEARS_BUILD": "建筑建造年份",
    "COMMONAREA": "公共区域面积",
    "ELEVATORS": "电梯指标",
    "ENTRANCES": "入口指标",
    "FLOORSMAX": "最高楼层",
    "FLOORSMIN": "最低楼层",
    "LANDAREA": "土地面积",
    "LIVINGAPARTMENTS": "居住公寓指标",
    "LIVINGAREA": "居住面积",
    "NONLIVINGAPARTMENTS": "非居住公寓指标",
    "NONLIVINGAREA": "非居住面积",
}

HOUSING_SUFFIX_CN = {
    "AVG": "均值口径",
    "MODE": "众数口径",
    "MEDI": "中位数口径",
}

EXCLUDED_COLUMNS = {"SK_ID_CURR", "TARGET", "APPLY_DATE", "dataset_flag"}
APPLICATION_TABLE = "application_{train|test}.csv"


def display_name_for(feature):
    """Return a business-facing Chinese name, or None for unknown features."""
    if feature in FEATURE_NAMES_CN:
        return FEATURE_NAMES_CN[feature]
    if feature.startswith("FLAG_DOCUMENT_"):
        n = feature[len("FLAG_DOCUMENT_"):]
        if n.isdigit():
            return "是否提交第{}类申请文件".format(n)
    for suffix, suffix_zh in HOUSING_SUFFIX_CN.items():
        ending = "_" + suffix
        if feature.endswith(ending):
            key = feature[:-len(ending)]
            if key in HOUSING_PARTS_CN:
                return "住房{}（{}）".format(HOUSING_PARTS_CN[key], suffix_zh)
    return None


def unit_for(feature):
    """Only infer units when semantics are explicit; unknown currency stays None."""
    if feature.startswith("DAYS_"):
        return "day"
    if feature == "OWN_CAR_AGE":
        return "year"
    if feature == "HOUR_APPR_PROCESS_START":
        return "hour"
    if (feature.startswith("CNT_") or feature.startswith("OBS_") or
            feature.startswith("DEF_") or
            feature.startswith("AMT_REQ_CREDIT_BUREAU_")):
        return "count"
    return None


def build_feature_metadata(columns, description_csv, verbose=True):
    """Build a MARS-compatible {feature: {display_name, description, data_source, unit}}.

    Args:
        columns: DataFrame column names (e.g. df.columns from a Polars DataFrame).
        description_csv: Official Kaggle HomeCredit_columns_description.csv path.
        verbose: Print match/missing stats.
    """
    columns = list(columns)
    active = set(columns) - EXCLUDED_COLUMNS
    metadata = {}

    with Path(description_csv).open("r", encoding="cp1252", newline="") as f:
        reader = csv.DictReader(f)
        required = {"Table", "Row", "Description", "Special"}
        missing_headers = required - set(reader.fieldnames or [])
        if missing_headers:
            raise ValueError("CSV missing columns: {}".format(sorted(missing_headers)))

        for record in reader:
            if (record.get("Table") or "").strip() != APPLICATION_TABLE:
                continue
            feature = (record.get("Row") or "").strip()
            if not feature or feature not in active:
                continue
            if feature in metadata:
                raise ValueError("Duplicate Home Credit feature: {}".format(feature))

            description = (record.get("Description") or "").strip() or None
            special = (record.get("Special") or "").strip()
            if special:
                description = "{} [Special: {}]".format(description or "", special)

            metadata[feature] = {
                "display_name": display_name_for(feature),
                "description": description,
                "data_source": "application",
                "unit": unit_for(feature),
            }

    uncovered = sorted(active - set(metadata))
    without_chinese_name = sorted(k for k, v in metadata.items() if not v["display_name"])
    if verbose:
        print("MARS metadata: {}/{} original features mapped".format(len(metadata), len(active)))
        if uncovered:
            print("Not in application description CSV:", uncovered)
        if without_chinese_name:
            print("Features without curated Chinese labels:", without_chinese_name)
    return metadata


def save_feature_metadata(metadata, path):
    """Save the metadata as UTF-8 JSON for reuse in another process."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")
    return path

In [ ]:
feature_metadata = build_feature_metadata(
    df.columns, CONFIG.RAW_DATA / "HomeCredit_columns_description.csv",
)
print(len(feature_metadata))
print(feature_metadata["EXT_SOURCE_2"])

In [ ]:
save_feature_metadata(feature_metadata, OUTPUT_DIR / "home_credit_feature_metadata.json")

# **3. 数据画像**

In [ ]:
PROFILE_METRICS = [
    "missing",
    "zeros",
    "unique",
    "mode",
    "psi",
    "mean",
    "std",
    "min",
    "p25",
    "median",
    "p75",
    "max",
]

profile_report = profile_stats(
    df,
    benchmark_df = df.filter(pl.col(seg)=='0train'),
    metrics=PROFILE_METRICS,
    features=fcols,
    feature_metadata=feature_metadata,
    group_col=seg,
    time_col=dt,
    time_grain="1m",
    missing_values=None,
    enable_sparkline=True,
    psi_include_missing=True,
    psi_include_special=False,
    psi_min_bin_size=0.01
)

In [ ]:
# 摘要含 show_overview / write_html / write_excel 的真实 API 用法。
profile_report

In [ ]:
profile_report.show_overview(sort_by="missing_rate", sort_ascending=True, limit=10)

In [ ]:
profile_report.show_trend('missing', limit=10, sort_ascending=True, sources="application")

In [ ]:
profile_report.show_trend('psi', limit=10, sort_ascending=False, sources="application")

In [ ]:
profile_report.overview_table

In [ ]:
profile_report.stats_tables["psi"]

In [ ]:
# 默认 schema 可从 describe() 核对；显式 columns 精确保持指定顺序。
assert "total" in profile_report.describe()["tables"]["stats.psi"]["fields"]
profile_report.get_table(
    "stats.psi",
    columns=["feature", "display_name", "dtype", "data_source", "0train", "1val", "2test", "3oot", "4_oot_undue", "total"],
    sort_by="4_oot_undue",
    filters={"4_oot_undue": {"op": "ge", "value": 0.25}},
    descending=True, limit=20,
)

In [ ]:
# HTML 的 DQ/Stats 内部指标页签、百分比、原值排序和色阶来自同一原生统计表。
profile_report.write_html(str(OUTPUT_DIR / "home_credit_2018_profile.html"))
profile_report.write_excel(str(OUTPUT_DIR / "home_credit_2018_profile.xlsx"))
profile_report.save(OUTPUT_DIR / "home_credit_profile.marsreport", overwrite=True)

In [ ]:
catalog = profile_report.describe()

print(catalog["report_id"])
print(catalog["tables"].keys())
print(catalog["parameters"])

In [ ]:
psi_alerts = profile_report.get_table(
    "stats.psi",
    filters={"4_oot_undue": {"op": "ge", "value": 0.25}},
    sort_by="4_oot_undue",
    descending=True,
    limit=30,
)

psi_alerts

In [ ]:
feature_evidence = profile_report.get_feature("AMT_REQ_CREDIT_BUREAU_MON")
# feature_evidence

In [ ]:
# 从异常结果中提取特征
alert_features = psi_alerts["feature"].to_list()

# 需要比较的数据集
group_cols = [
    "0train", "1val", "2test",
    "3oot", "4_oot_undue", "total"
]

# 针对异常特征查询各类证据
queries = {
    "stats.psi": {
        "columns": ["feature", "display_name", "dtype", "data_source", *group_cols],
        "sort_by": "4_oot_undue",
        "descending": True,
        "limit": 30,
    },
    "dq.missing": {
        "columns": ["feature", *group_cols],
        "limit": 30,
    },
    "dq.zeros": {
        "columns": ["feature", *group_cols],
        "limit": 30,
    },
    "stats.mean": {
        "columns": ["feature", *group_cols],
        "limit": 30,
    },
    "stats.median": {
        "columns": ["feature", *group_cols],
        "limit": 30,
    },
    "overview": {
        "columns": [
            "feature", "display_name", "dtype", "data_source",
            "missing_rate", "zeros_rate",
            "unique_rate", "mode_rate",
            "mean", "std",
        ],
        "limit": 30,
    },
}

# 如果有计算状态表，一并提供
if "calculation_status" in profile_report.describe()["tables"]:
    queries["calculation_status"] = {
        "limit": 30,
    }

# 生成结构化 Agent 上下文
context_json = profile_report.to_ai_context(
    features=alert_features,
    queries=queries,
    max_chars=24000,
)

context = json.loads(context_json)
print(json.dumps(context, ensure_ascii=False, indent=2))

In [ ]:
# OOT 的 PSI
psi = profile_report.get_table(
    "stats.psi",
    columns=["feature", "4_oot_undue"],
).rename({"4_oot_undue": "psi_oot"})

# Train 和 OOT 缺失率
missing = profile_report.get_table(
    "dq.missing",
    columns=["feature", "0train", "4_oot_undue"],
).rename({
    "0train": "missing_train",
    "4_oot_undue": "missing_oot",
})

# 建立异常候选表
alerts = (
    psi.join(missing, on="feature", how="left")
    .with_columns(
        (pl.col("missing_oot") - pl.col("missing_train"))
        .alias("missing_delta")
    )
    .filter(
        (pl.col("psi_oot") >= 0.25)
        | (pl.col("missing_delta") >= 0.05)
    )
    .sort("psi_oot", descending=True)
)

alerts

# **4. 特征分箱**

In [ ]:

risk_profile = profile_risk(
    df,
    benchmark_df = df.filter(pl.col(seg)=='0train'),
    target=[y],
    amount_col=amt,
    features=fcols,
    feature_metadata=feature_metadata,
    group_col=seg,
    time_col=dt,
    # time_grain="1m",
    missing_values = None,
    psi_include_missing=True,
    psi_include_special=False,
    binning_type="native", # "native"
    method = "cart", # "quantile", "uniform"
    n_bins = 10,
    min_bin_size=0.05,
    monotonic_trend = "auto_asc_desc",
    # binner_params={
        # "merge_small_bins": True,
        # "remove_empty_bins": True
    # },
    ordered_metric_sort_by="woe",
    ks_method = 'binned'
)
risk_report = risk_profile.report
# risk_month_report.write_excel(engine="xlwings")
# risk_month_report.write_html()

In [ ]:
# 新报告不再重复生成 feature 维度的总体 trend.bad_rate；箱坏率保留在 detail。
risk_report

In [ ]:
risk_report.summary_table

In [ ]:
risk_report.show_summary(sort_by="iv", sort_ascending=False, limit=10)

In [ ]:
risk_report.get_table(
    "summary",
    sort_by="ks",
    descending=True,
    limit=10,
)

In [ ]:
risk_report.show_trend(
    "ks",
    sort_by="0train",
    sort_ascending=False,
    limit=10,
) 

In [ ]:
risk_report.get_table(
    "trend.psi",
    filters={
        "4_oot_undue": {"op": "ge", "value": 0.1}
    },
    sort_by="4_oot_undue",
    descending=True,
    limit=5,
)

In [ ]:
feature = "AMT_REQ_CREDIT_BUREAU_MON"

risk_report.get_table(
    "detail",
    features=feature,
    filters={
        "mars_group": "0train",
        "y": y,
    },
    columns=[
        "feature", "display_name", "dtype", "data_source",
        "mars_group",
        "bin_index",
        "bin_label",
        "count",
        "observed_count",
        "pct",
        "bad_rate",
        "lift",
        "iv_bin",
        "psi_bin",
        "bin_type",
    ],
    sort_by="bin_index",
)

In [ ]:
risk_report.get_table(
    "detail",
    features="AMT_REQ_CREDIT_BUREAU_MON",
    filters={"mars_group": "4_oot_undue"},
    columns=[
        "feature", "display_name", "dtype", "data_source", "bin_label", "count",
        "pct", "psi_bin", "bad_rate",
    ],
    sort_by="psi_bin",
    descending=True,
)

In [ ]:
# 中文名位于原说明框首行；显式分组面板显示本次全样本的真实日期。
risk_report.plot_risk_trends(
    features="EXT_SOURCE_2", show_risk="count", include_total=True, dpi=150,
)

In [ ]:
# 隐藏 Total 只改变面板显示，全局统计与 RC 参考仍保留。
risk_report.plot_risk_trends(
    features="EXT_SOURCE_2", show_risk="count", include_total=False, dpi=150,
)

In [ ]:
for mode in ("count", "amt", "both"):
    risk_report.plot_risk_trends(
        features="EXT_SOURCE_2", show_risk=mode, include_total=False, dpi=300,
    )
risk_report.save_risk_trend_images(
    OUTPUT_DIR / "with_total", features="EXT_SOURCE_2", image_format="png",
    show_risk="both", include_total=True, dpi=300,
)
risk_report.save_risk_trend_images(
    OUTPUT_DIR / "without_total", features="EXT_SOURCE_2", image_format="png",
    show_risk="both", include_total=False, dpi=300,
)
risk_report.save_risk_trend_images(
    OUTPUT_DIR / "svg", features="EXT_SOURCE_2", image_format="svg",
    show_risk="both", include_total=False, dpi=300,
)

In [ ]:
alert_features = [
    "AMT_REQ_CREDIT_BUREAU_MON",
    "AMT_REQ_CREDIT_BUREAU_QRT",
]

context_json = risk_report.to_ai_context(
    features=alert_features,
    queries={
        "summary": {
            "columns": [
                "feature", "display_name", "dtype", "data_source", "iv", "ks", "auc",
                "psi_max", "rc_min", "missing",
            ],
            "limit": 10,
        },
        "trend.psi": {
            "limit": 10,
        },
        "trend.ks": {
            "limit": 10,
        },
        "trend.iv": {
            "limit": 10,
        },
        "trend.missing": {
            "limit": 10,
        },
        "detail": {
            "filters": {
                "mars_group": {
                    "op": "in",
                    "value": ["0train", "3oot", "4_oot_undue"],
                }
            },
            "columns": [
                "y", "feature", "display_name", "dtype", "data_source", "mars_group",
                "bin_index", "bin_label",
                "count", "observed_count",
                "pct", "bad_rate", "psi_bin",
                "bin_type",
            ],
            "limit": 100,
        },
    },
    max_chars=36000,
)

context = json.loads(context_json)
print(json.dumps(context, ensure_ascii=False, indent=2))

In [ ]:
risk_report.write_html(
    str(OUTPUT_DIR / "home_credit_risk_binned.html"), max_plots=3,
    include_total=False, chart_embed_mode="inline",
)
# 本机实际 Excel 模板交互验收；没有 Office 时请显式选择 openpyxl 并记录限制。
risk_report.write_excel(str(OUTPUT_DIR / "home_credit_risk_binned.xlsx"), engine="xlwings")
binned_report = risk_report

In [ ]:
from mars.reporting import snapshot_report, load_report

snapshot_report(risk_report).write_excel(str(OUTPUT_DIR / "home_credit_risk_static.xlsx"))
risk_report.save(OUTPUT_DIR / "home_credit_risk_binned.marsreport", overwrite=True)
restored = load_report(OUTPUT_DIR / "home_credit_risk_binned.marsreport")
restored_profile = load_report(OUTPUT_DIR / "home_credit_profile.marsreport")
restored_profile.write_html(str(OUTPUT_DIR / "home_credit_profile_restored.html"))
assert restored.report_id == risk_report.report_id
assert restored.get_table("summary").equals(risk_report.get_table("summary"))
restored.show_table("trend.ks", columns=["feature", "display_name", "dtype", "data_source", "0train", "3oot", "Total"], limit=3)

In [ ]:
binner = risk_profile.binner

# 查看拟合诊断
binner.get_fit_report()

# 查看某个特征的箱标签映射
binner.get_bin_mapping("EXT_SOURCE_2")

# 保存分箱规则
binner.save_json(OUTPUT_DIR / "home_credit_binner.json")

In [ ]:
# fcols

In [ ]:
raw_profile = profile_risk(
    df,
    benchmark_df = df.filter(pl.col(seg)=='0train'),
    target=[y],
    amount_col=amt,
    features=[
        'EXT_SOURCE_1',
        'EXT_SOURCE_2',
        'EXT_SOURCE_3',
    ],
    feature_metadata=feature_metadata,
    group_col=seg,
    time_col=dt,
    # time_grain="1m",
    missing_values = None,
    psi_include_missing=True,
    psi_include_special=False,
    binning_type="native", # "native"
    method = "cart", # "quantile", "uniform"
    n_bins = 10,
    min_bin_size=0.05,
    monotonic_trend = "auto_asc_desc",
    # binner_params={
        # "merge_small_bins": True,
        # "remove_empty_bins": True
    # },
    ordered_metric_sort_by="bin_index",
    ks_method = 'raw'
)
raw_report = raw_profile.report

# 最终 raw KS 和辅助箱级 ks_bin 分别导出，文件与 binned 报告独立。
raw_report.write_excel(str(OUTPUT_DIR / "home_credit_risk_raw_ks.xlsx"), engine="xlwings")
raw_report.save(OUTPUT_DIR / "home_credit_risk_raw_ks.marsreport", overwrite=True)
raw_ks = raw_report.get_table("summary", columns=["feature", "display_name", "dtype", "data_source", "ks"])
binned_ks = binned_report.get_table("summary", features=["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"], columns=["feature", "ks"]).rename({"ks": "ks_binned"})
raw_ks.rename({"ks": "ks_raw"}).join(binned_ks, on="feature", how="left")

In [ ]:
# benchmark_df 用于分箱拟合/PSI 基准；fit(df) 的筛选仍覆盖传入全样本，Test/OOT 参与本例筛选。
ms_selector = MarsStatsSelector(
    # ================= 数据质量与异常值 =================
    missing_thr=0.90,
    zeros_thr=0.90,
    mode_thr=0.90,
    missing_values=None,
    special_values=None,

    # ================== 粗筛阶段配置 ==================
    skip_rough_scan=False,
    rough_iv_thr=0.02,
    rough_lift_thr=1.2,
    rough_min_sample_rate=0.02,
    rough_binning_params={
        "n_bins": 10,
    },

    # ================== 精筛阶段配置 ==================
    skip_fine_scan=True,
    iv_thr=0.02,
    lift_thr=1.2,
    min_sample_rate=0.05,
    binning_params={
        "prebinning_method": "cart",
        "n_bins": 10,
    },

    # =============== 稳定性与相关性约束 ===============
    psi_thr=0.05,
    rc_thr=0.75,
    corr_thr=0.9,
    psi_include_missing=True,
    psi_include_special=False,

    # =================== 运行与性能 ===================
    batch_size=100,
    n_jobs=-1,
)
ms_selector.fit(
    df,
    benchmark_df=df.filter(pl.col(seg) == "0train"),
    target=y,
    features=fcols,
    feature_metadata=feature_metadata,
    group_col=seg,
    time_col=dt,
    # time_grain="1m",
)
selected_features = list(ms_selector.selected_features_)

In [ ]:
# 原生审计保留多阶段事件；审计行数不等于候选特征数。
selection_report = ms_selector.get_report()
selection_report

In [ ]:
selection_report.filter(
    pl.col("status") == "Dropped"
)

In [ ]:
selection_report.filter(
    pl.col("status") == "Dropped"
).group_by(
    ["stage", "reason"]
).agg(
    pl.col("feature").n_unique().alias("feature_count")
).sort(
    "feature_count",
    descending=True
)

In [ ]:
ms_selector.show_summary()

In [ ]:
corr_report = ms_selector.get_correlation_report()

print(list(corr_report.describe()["tables"]))

In [ ]:
corr_report.get_table(
    "pairs",
    sort_by="abs_correlation",
    descending=True,
    limit=30,
)

In [ ]:
corr_report.get_table(
    "correlation_decisions",
    limit=50,
)